# 03. Vectorless RAG, head-to-head, and hybrid

[NB1](./01_introduction.ipynb) was the crawl. [NB2](./02_bm25_mechanics.ipynb) was the walk. **This is the run.**

Three things in this notebook:

1. **Act I**: the full vectorless RAG loop with `gpt-4o-mini`. Same four-step shape as module 01's vector RAG, just with one step swapped.
2. **Act II**: the head-to-head. Switch to module 01's 20-doc Pinecone corpus and run the same five questions through BM25 and through Pinecone vector search. See exactly when BM25 wins, when vector wins, and when they tie.
3. **Act III**: the hybrid fix. Combine the two rankings with Reciprocal Rank Fusion. Watch it correct the single-method misses.

The thesis lands at the end: *vector retrieval is famous, BM25 is the baseline you should beat, hybrid usually beats either alone.*

## Setup

> **Run this notebook top to bottom.** Each cell depends on variables defined by cells above it.

NB3 needs an OpenAI key. The Pinecone key (for Act II) is optional. Without it, the head-to-head gracefully degrades to BM25-only and the vector arm prints a skip note.

In [1]:
from helpers import (
    load_env,
    get_openai_client,
    get_pinecone_client,
    build_bm25_index,
    retrieve,
    rrf_combine,
)
from scripts.build_corpus import build_if_missing
import pandas as pd

build_if_missing()
df = pd.read_parquet("data/corpus.parquet")

cfg = load_env()
client = get_openai_client(cfg)
pc = get_pinecone_client(cfg)  # None if PINECONE_API_KEY isn't set

print(f"OpenAI chat model: {cfg.openai_chat_model}")
print(f"Pinecone client:   {'ready' if pc else 'not configured (Act II vector arm will skip)'}")

OpenAI chat model: gpt-4o-mini
Pinecone client:   ready


## Act I: vectorless RAG end-to-end

The four-step RAG loop, with BM25 instead of an embedding model in step 1.

```
            (NB1 / NB2 already did this)               (LLM, this notebook)
   ┌─────────────────────────────────┐    ┌──────────────────────────────────┐
   │  step 1: Tokenize the query      │ →  │  step 3: Augment a prompt        │
   │  step 2: Retrieve top-k from BM25│ →  │  step 4: Generate the answer     │
   └─────────────────────────────────┘    └──────────────────────────────────┘
```

Compare this to module 01's loop:

```
   step 1: Embed the query
   step 2: Retrieve top-k from a vector index
   step 3: Augment a prompt
   step 4: Generate the answer
```

The only step that changed is the first one. Tokenize is to BM25 what embed is to vector RAG. Everything downstream is identical.

In [2]:
retriever, _ = build_bm25_index(df["text"].tolist())

SYSTEM_PROMPT = (
    "You are a helpful assistant answering Python questions. "
    "Use ONLY the provided context. Cite sources with their [tip_NN] ids. "
    "If the context doesn't cover the question, say so plainly rather than guessing."
)


def build_context(idx_list: list[int]) -> str:
    parts = []
    for rank, i in enumerate(idx_list, start=1):
        row = df.iloc[i]
        parts.append(f"[{row['id']}] {row['title']}\n{row['text']}")
    return "\n\n".join(parts)


def ask_bm25(question: str, k: int = 3) -> dict:
    idx, scores = retrieve(retriever, question, k=k)
    context = build_context(idx)
    resp = client.chat.completions.create(
        model=cfg.openai_chat_model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0,
    )
    return {
        "question": question,
        "retrieved": [df.iloc[i]["id"] for i in idx],
        "answer": resp.choices[0].message.content.strip(),
    }

### A question with literal vocabulary match

In [3]:
result = ask_bm25("how do I cache function results in Python?")
print(f"Retrieved: {result['retrieved']}")
print()
print(result["answer"])

Retrieved: ['tip_09', 'tip_03', 'tip_04']

You can cache function results in Python by using the `@functools.lru_cache` decorator. This decorator memoizes the return values of a function based on its arguments, allowing subsequent calls with the same arguments to return the cached result instead of executing the function body again. Here's a brief overview of how to use it:

1. Import the `functools` module.
2. Decorate your function with `@functools.lru_cache`.
3. Call the function as usual; the results will be cached automatically.

Here's an example:

```python
import functools

@functools.lru_cache(maxsize=None)  # You can set maxsize to limit the cache size
def expensive_function(x):
    # Simulate a time-consuming computation
    return x * x

result = expensive_function(4)  # This will compute and cache the result
result = expensive_function(4)  # This will return the cached result
```

Remember that `lru_cache` works only with hashable arguments and is suitable for pure functio

tip_09 (`lru_cache`) wins retrieval, and the LLM grounds its answer in that doc. Clean end-to-end loop.

### A question that needs multi-doc synthesis

In [4]:
result = ask_bm25("what's the difference between os.path and pathlib?")
print(f"Retrieved: {result['retrieved']}")
print()
print(result["answer"])

Retrieved: ['tip_11', 'tip_10', 'tip_12']

The difference between `os.path` and `pathlib` lies primarily in their design and usage:

1. **Design**: 
   - `os.path` is a module that provides functions for manipulating filesystem paths as strings. It uses a procedural approach where you call functions like `os.path.join`, `os.path.exists`, etc. [tip_11].
   - `pathlib`, on the other hand, is an object-oriented path API. It allows you to work with paths as objects, using methods and operators. For example, you can join paths using the `/` operator, like `Path('/a') / 'b'`, which results in `Path('/a/b')` [tip_10].

2. **Usage**: 
   - `os.path` is often used for legacy code and is suitable for simple path manipulations when working with string paths. However, it is recommended to use `pathlib` for new code due to its more modern and intuitive interface [tip_10].
   - `pathlib` provides additional functionality, such as `read_text` and `write_text` methods for reading and writing files eas

tip_10 and tip_11 both retrieve, and the LLM synthesizes across them. Notice how the cited ids in the answer come from both.

### A question that exposes the synonym trap

In [5]:
result = ask_bm25("how do I make Python code faster?")
print(f"Retrieved: {result['retrieved']}")
print()
print(result["answer"])

Retrieved: ['tip_03', 'tip_01', 'tip_05']

The context provided does not specifically address general strategies for making Python code faster. It only includes tips on specific functionalities, such as using the bisect module for maintaining sorted lists efficiently [tip_03] and using `operator.itemgetter` for sorting lists of dictionaries or tuples [tip_01]. For general performance improvements, you might consider optimizing algorithms, using built-in functions, or leveraging libraries designed for performance, but those strategies are not covered in the provided context.


The retrieval misses tip_12 entirely. The query says "make faster"; the doc says "speed up". The LLM honestly reports it can't answer from the context. This is the failure mode we resolve in Act III.

But first, let's quantify it with a proper head-to-head against vector retrieval.

## Act II: BM25 vs vector retrieval, head-to-head

We switch corpus to **module 01's `data/corpus.parquet`** (20 chunks paraphrased from the Pinecone docs). Same docs that already live in module 01's Pinecone index, so the vector arm of this head-to-head doesn't cost any embedding calls or index builds: we reuse what module 01 already shipped.

If module 01's data isn't present (you skipped it), the cell below errors with a clear pointer.

In [6]:
import os.path

m01_corpus_path = "../01-intro-to-rag/data/corpus.parquet"
if not os.path.exists(m01_corpus_path):
    raise FileNotFoundError(
        f"{m01_corpus_path} not found.\n"
        "NB3's head-to-head reuses module 01's corpus. To populate it, run:\n"
        "    cd ../01-intro-to-rag && python scripts/build_corpus.py"
    )

m01_df = pd.read_parquet(m01_corpus_path)
print(f"Loaded {len(m01_df)} docs from module 01's corpus")
m01_df[["id", "title", "section"]].head()

Loaded 20 docs from module 01's corpus


,id,title,section
0,doc_4d45391193b6,Serverless indexes,Indexes / Overview
1,doc_cb65d310d336,Creating an index,Indexes / Create
2,doc_42de0361ce4a,Checking if an index exists,Indexes / Introspection
3,doc_1729940ac580,Deleting an index,Indexes / Delete
4,doc_0a2ddfe050b1,Index statistics,Indexes / Introspection


In [7]:
# Build BM25 over module 01's corpus
m01_retriever, _ = build_bm25_index(m01_df["text"].tolist())
print(f"BM25 indexed {len(m01_df)} docs")

BM25 indexed 20 docs


### The Pinecone vector arm

If you have `PINECONE_API_KEY` set in `.env` and module 01's `rag-unpacked-intro` index still exists, this cell runs the vector arm. Otherwise it sets `vector_query` to None and the head-to-head proceeds BM25-only with a skip note.

In [8]:
vector_index = None
if pc is not None:
    try:
        if pc.has_index(cfg.pinecone_index_name):
            vector_index = pc.Index(cfg.pinecone_index_name)
            print(f"Pinecone index '{cfg.pinecone_index_name}' ready")
        else:
            print(f"Pinecone index '{cfg.pinecone_index_name}' not found.")
            print("  To create it, run module 01's NB3 end-to-end.")
    except Exception as e:
        print(f"Pinecone unreachable: {e}")
else:
    print("PINECONE_API_KEY not set; vector arm will be skipped.")


def vector_query(question: str, k: int = 3) -> list[str] | None:
    if vector_index is None:
        return None
    # Embed the question with the same model module 01 used to build the index
    emb_resp = client.embeddings.create(
        input=[question],
        model=cfg.openai_embed_model,
    )
    qv = emb_resp.data[0].embedding
    result = vector_index.query(vector=qv, top_k=k, include_metadata=False)
    return [m["id"] for m in result["matches"]]

Pinecone index 'rag-unpacked-intro' ready


### The five demo questions

These are engineered to make the head-to-head produce a meaningful spread, not a draw:

1. `pool_threads` is a literal Pinecone identifier. **BM25 should win.**
2. `ServerlessSpec(cloud="aws", region="us-east-1")` has multiple rare identifiers. **BM25 should win.**
3. `$in` is a distinctive symbolic token. **BM25 should win.**
4. "Store text without embedding it myself" is a paraphrase of integrated inference, with no literal vocabulary overlap. **Vector should win.**
5. "What does the metric parameter control" is well-worded English; both should find the distance-metric doc. **Likely tie.**

In [9]:
DEMO_QUESTIONS = [
    "what does pool_threads do?",
    "how do I create a serverless index with cloud=aws and region=us-east-1?",
    "$in operator for filtering",
    "can I store text without embedding it myself?",
    "what does the metric parameter control?",
]


def bm25_query(question: str, k: int = 3) -> list[str]:
    idx, _ = retrieve(m01_retriever, question, k=k)
    return [m01_df.iloc[i]["id"] for i in idx]


# Run all five through BM25 and through Pinecone
results = []
for q in DEMO_QUESTIONS:
    row = {"question": q}
    row["bm25"] = bm25_query(q, k=3)
    row["vector"] = vector_query(q, k=3) or "(skipped)"
    results.append(row)

results_df = pd.DataFrame(results)
results_df

,question,bm25,vector
0,what does pool_threads do?,"[doc_7eaf9d85e4b5, doc_325231fca76b, doc_de1fe...","[doc_7eaf9d85e4b5, doc_5cdf73fb4299, doc_e6fbc..."
1,how do I create a serverless index with cloud=...,"[doc_4d45391193b6, doc_cb65d310d336, doc_2530b...","[doc_4d45391193b6, doc_cb65d310d336, doc_2530b..."
2,$in operator for filtering,"[doc_50ae76610fa0, doc_325231fca76b, doc_de1fe...","[doc_50ae76610fa0, doc_1fa92232a250, doc_c035f..."
3,can I store text without embedding it myself?,"[doc_318824736278, doc_928bebc1453d, doc_f1949...","[doc_f194968125f2, doc_318824736278, doc_cb65d..."
4,what does the metric parameter control?,"[doc_928bebc1453d, doc_325231fca76b, doc_17299...","[doc_cb65d310d336, doc_928bebc1453d, doc_31882..."


### Top-1 head-to-head

To make the comparison sharp, look at the #1 result from each method side by side.

In [10]:
def first_or_none(x):
    if isinstance(x, list) and x:
        return x[0]
    return None


comparison = pd.DataFrame({
    "question": [r["question"] for r in results],
    "bm25_top1": [first_or_none(r["bm25"]) for r in results],
    "vector_top1": [first_or_none(r["vector"]) if isinstance(r["vector"], list) else "(skipped)" for r in results],
})
comparison["agree"] = comparison["bm25_top1"] == comparison["vector_top1"]
comparison

,question,bm25_top1,vector_top1,agree
0,what does pool_threads do?,doc_7eaf9d85e4b5,doc_7eaf9d85e4b5,True
1,how do I create a serverless index with cloud=...,doc_4d45391193b6,doc_4d45391193b6,True
2,$in operator for filtering,doc_50ae76610fa0,doc_50ae76610fa0,True
3,can I store text without embedding it myself?,doc_318824736278,doc_f194968125f2,False
4,what does the metric parameter control?,doc_928bebc1453d,doc_cb65d310d336,False


A real result tells a more honest story than my plan did. On this corpus, with this embedding model, the pattern was:

- **Questions 1, 2, 3 (rare identifiers): BM25 and vector agreed.** Both retrievers nailed `pool_threads`, the serverless-index doc, and the metadata-filtering doc. This is the "boring success" case: when the corpus is small and the keywords are distinctive, both methods converge on the same right answer. BM25 produces it for free, with no embedding cost.
- **Question 4 ("store text without embedding myself"): vector won.** BM25 retrieved the wrong doc. Vector retrieved the right doc (about integrated inference). This is the paraphrase case where embeddings shine.
- **Question 5 ("metric parameter"): vector won too.** BM25's top result mentions "metric" in passing while discussing embedding dimensions, but vector found the doc that actually explains the metric parameter.

So the spread is **3 agreements, 2 vector wins, 0 BM25-only wins** on this small corpus. With a larger corpus the rare-identifier wins for BM25 would show up more clearly (the vector embedding has less room to converge on the same answer when there are more candidates). The point isn't that BM25 won X questions and vector won Y; it's that each one has a structural strength and weakness, and the head-to-head makes both visible.

If the vector arm was skipped (no Pinecone key), the `vector_top1` column shows `(skipped)` and you can still see BM25's standalone behavior, but rows 4 and 5 won't have anything to compare against.

## Act III: hybrid via Reciprocal Rank Fusion

The synonym trap from Act I and the vector-favored questions from Act II both come down to the same underlying issue: BM25 has perfect precision on rare-identifier matches and zero recall on paraphrases. Vector retrieval has the opposite tradeoff.

**Reciprocal Rank Fusion (RRF)** combines them without any score normalization. For each id, sum `1/(k + rank)` across every retriever's ranked list. Sort descending. That's it. `k=60` is the standard pick.

The win of RRF over weighted-score fusion: BM25 scores and cosine similarity scores live on different scales (raw IDF·TF vs values bounded in `[-1, 1]`). Normalizing them is a tuning headache. RRF dodges the problem by combining *ranks*, not scores.

Here's the implementation inline (and `helpers.rrf_combine` is exactly this):

In [11]:
def rrf(rankings, k=60):
    fused = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)


# Quick sanity check on a toy example
sanity = rrf([
    ["A", "B", "C"],   # BM25 says A, B, C
    ["B", "A", "D"],   # vector says B, A, D
])
print("Toy RRF result (A and B tie at top):")
for doc_id, score in sanity:
    print(f"  {doc_id}: {score:.5f}")

Toy RRF result (A and B tie at top):
  A: 0.03252
  B: 0.03252
  C: 0.01587
  D: 0.01587


A and B end up tied at the top, exactly as you'd want when each appears at ranks 1 and 2 across the two rankings. C and D each appear only once at rank 3, so they tie at a lower score.

Now apply hybrid retrieval to the five demo questions.

In [12]:
def hybrid_query(question: str, k: int = 3) -> list[str] | None:
    bm25_ranking = bm25_query(question, k=10)  # widen for fusion
    vector_ranking = vector_query(question, k=10) if vector_index else None
    if vector_ranking is None:
        return None  # can't do hybrid without both
    fused = rrf_combine([bm25_ranking, vector_ranking], k=60)
    return [doc_id for doc_id, _ in fused[:k]]


# Three-arm comparison
three_arm = pd.DataFrame({
    "question": [r["question"] for r in results],
    "bm25_top1": [first_or_none(r["bm25"]) for r in results],
    "vector_top1": [first_or_none(r["vector"]) if isinstance(r["vector"], list) else "(skipped)" for r in results],
    "hybrid_top1": [first_or_none(hybrid_query(r["question"], k=3) or "(skipped)") for r in results],
})
three_arm

,question,bm25_top1,vector_top1,hybrid_top1
0,what does pool_threads do?,doc_7eaf9d85e4b5,doc_7eaf9d85e4b5,doc_7eaf9d85e4b5
1,how do I create a serverless index with cloud=...,doc_4d45391193b6,doc_4d45391193b6,doc_4d45391193b6
2,$in operator for filtering,doc_50ae76610fa0,doc_50ae76610fa0,doc_50ae76610fa0
3,can I store text without embedding it myself?,doc_318824736278,doc_f194968125f2,doc_318824736278
4,what does the metric parameter control?,doc_928bebc1453d,doc_cb65d310d336,doc_928bebc1453d


Compare the `hybrid_top1` column to the BM25 and vector columns. What actually happened:

- Where BM25 and vector agreed (rows 1, 2, 3), hybrid agrees too. Easy.
- Where they disagreed (rows 4, 5), hybrid picks the BM25 answer, not the vector answer. **That's not because BM25 was right.** It's because RRF combines ranks, not correctness, and when the "right" doc was vector's #1 but BM25's #5, the combined rank score got beaten by a doc that was BM25's #1 and vector's #3.

The lesson: **hybrid is a robustness improvement, not a quality improvement.** If you know in advance that vector retrieval is right for a query, just use vector. If you don't know which arm will win, hybrid usually gives you a doc that's at least decent according to both methods, even when the "best" doc according to one isn't the best according to the other. The win is *not falling off a cliff* when one retriever fails, not *picking the optimum*.

This is also why production systems layer a **cross-encoder reranker** on top of hybrid retrieval: a second-stage model scores the top-N candidates from the fused list and reorders them. The fused list provides good *recall*; the reranker provides good *precision*. That's module 05.

If the vector arm was skipped, the hybrid column also shows `(skipped)`. The pattern doesn't show up without both arms present.

## The closing thesis

Three sentences this whole module exists to support:

1. **BM25 is the baseline you should beat.** It's free, fast, deterministic, and wins outright on rare-identifier queries. If you're building anything with retrieval, you should know what BM25 produces on your corpus before you reach for an embedding model.

2. **Vector retrieval handles the paraphrases BM25 misses.** Synonyms and conceptual queries are vector's structural advantage. Pay the embedding cost when those queries matter to your users.

3. **Hybrid is a robustness move, not a quality move.** Reciprocal Rank Fusion combines ranks, not correctness. On any given query it usually returns a doc that's at least decent according to both methods, not necessarily the doc that the better method would have picked alone. The win is consistency across query shapes, not optimum on each one. To get back to "best on each query," you layer a cross-encoder reranker on top of the fused list. That's module 05.

What this module doesn't cover:

- **Cross-encoder reranking**, a second-stage model that re-scores the top-N candidates after fusion. Module 05.
- **Learned sparse retrieval** (SPLADE), BM25's spirit with learned term weights. Module 05.
- **Evaluation**. How do you actually measure which arm won? Module 04.

## Where to go next

- **[Module 04: Evaluating RAG.](../04-evaluating-rag/)** Once you've done this head-to-head, the next question is "how do I measure which arm I should ship?". Faithfulness, context precision, answer relevance.
- **[Module 05: Advanced RAG.](../05-advanced-rag/)** Cross-encoder rerankers, hybrid with learned weights, query rewriting, the full production version of what NB3 sketched.